# Figure 2 — CLAMP recovers interpretable, cell-type-specific gene modules

Each panel loads its own precomputed data and builds its plot natively in
this notebook (not a rendered PNG from another notebook) so panels compose
at a consistent scale without raster stretching. Heavy computation (model
scoring, enrichment analysis) still lives in the upstream analysis
notebooks; this notebook only reads their already-computed CSV outputs.

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(yaml)
  library(here)
})


## Styling

In [ ]:
AXIS_TITLE_SIZE  <- 10.5
AXIS_LABEL_SIZE  <- 8
PLOT_TITLE_SIZE  <- 9
LEGEND_TEXT_SIZE <- 7
LEGEND_TITLE_SIZE <- 8

LABEL_SIZE <- 10
LABEL_FONTFACE <- "bold"
LABEL_FONTFAMILY <- "Helvetica"
CLAMP_COLOR <- "#0072B2"

DATASET_LABELS <- c(
  Brain_Mathys2023 = "Brain Mathys",
  Brain_Xiong2023  = "Brain Xiong",
  Heart_Datar2026  = "Heart Datar",
  PBMC_1k1k        = "PBMC 1k1k",
  PBMC_Perez2022   = "PBMC Perez",
  Lung_Sikkema2023 = "Lung Sikkema"
)

# Distinct from MODEL_COLORS (used for methods in panel A) -- a separate
# qualitative palette (ColorBrewer Dark2) for the 6 datasets in panel B.
DATASET_COLORS <- c(
  "PBMC 1k1k"    = "#1B9E77",
  "PBMC Perez"   = "#D95F02",
  "Heart Datar"  = "#7570B3",
  "Brain Xiong"  = "#E7298A",
  "Brain Mathys" = "#66A61E",
  "Lung Sikkema" = "#E6AB02"
)

# Truncate long "<cell type> - LV##" axis labels so they do not overlap
# when many categories are packed into a narrow dot-grid panel.

# Standard abbreviation; "precursor"/"progenitor" both show up upstream
# for the same cell type.
abbreviate_ct <- function(x) {
  x <- gsub("Oligodendrocyte [Pp]rogenitor [Cc]ells?", "OPC", x)
  x <- gsub("Oligodendrocyte [Pp]recursor [Cc]ells?", "OPC", x)
  x
}

# Wrap (not truncate) long labels onto multiple lines, so the full text
# stays readable instead of being cut off with "...".
wrap_label <- function(x, width = 14) {
  vapply(x, function(s) paste(strwrap(s, width = width), collapse = "\n"),
         character(1), USE.NAMES = FALSE)
}
shorten_label <- function(x, width = 18) {
  ifelse(nchar(x) > width, paste0(substr(x, 1, width - 1), "\u2026"), x)
}

theme_pub <- function(base_size = 7, base_line_size = 0.3) {
    theme_classic(base_size = base_size, base_family = "Helvetica") %+replace%
    theme(
        plot.title        = element_text(size = PLOT_TITLE_SIZE, face = "plain", hjust = 0.5,
                                          margin = margin(b = 2)),
        axis.line         = element_line(linewidth = base_line_size),
        axis.ticks        = element_line(linewidth = base_line_size),
        axis.text         = element_text(size = AXIS_LABEL_SIZE),
        axis.title        = element_text(size = AXIS_TITLE_SIZE, face = "plain"),
        legend.text       = element_text(size = LEGEND_TEXT_SIZE),
        legend.title      = element_text(size = LEGEND_TITLE_SIZE, face = "plain"),
        legend.key.size   = unit(3, "mm"),
        legend.background = element_blank(),
        legend.key        = element_blank(),
        panel.grid        = element_blank(),
        strip.text        = element_text(size = PLOT_TITLE_SIZE, face = "plain"),
        strip.background  = element_blank(),
        plot.margin       = unit(c(1, 2, 1, 1), "mm")
    )
}

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS_RAW <- unlist(cfg$MODEL_COLORS)
names(MODEL_COLORS_RAW)[names(MODEL_COLORS_RAW) == "GenomicSuperSignature"] <- "GSSig"


## Panel A: pooled benchmark boxplot

In [ ]:
long     <- fread(snakemake@input[["benchmark_long"]])
long_box <- long[truth == "v0" & !is.na(cor)]

mean_order <- long_box[, .(m = mean(cor, na.rm = TRUE)), by = method]
setorder(mean_order, -m)
method_order <- as.character(mean_order$method)
long_box[, method := factor(method, levels = method_order)]

METHOD_COLORS <- MODEL_COLORS_RAW[method_order]
METHOD_COLORS[is.na(METHOD_COLORS)] <- "grey70"
names(METHOD_COLORS) <- method_order

mean_df <- long_box[, .(mean_cor = mean(cor, na.rm = TRUE),
                        max_cor  = max(cor, na.rm = TRUE)), by = method]
mean_df[, method := factor(method, levels = method_order)]

xpos <- setNames(seq_along(method_order), method_order)
other_methods <- setdiff(method_order, "CLAMPfull")

fmt_q <- function(q) {
  if (q >= 0.001) {
    sprintf('"%.3f"', q)
  } else {
    e_str    <- formatC(q, format = "e", digits = 1)
    parts    <- strsplit(e_str, "e")[[1]]
    mantissa <- trimws(parts[1])
    exp_val  <- as.integer(parts[2])
    sprintf('%s%%*%%10^{%d}', mantissa, exp_val)
  }
}

wide_ct_box <- dcast(long_box, dataset + cell_type ~ method, value.var = "cor")

comp_df_paired <- rbind(
  data.table(a = "CLAMPfull", b = other_methods[1:3]),
  data.table(a = "CLAMPbase", b = "PLIER")
)
comp_df_paired[, p_raw := mapply(function(a, b) {
  d <- wide_ct_box[!is.na(get(a)) & !is.na(get(b))]
  wilcox.test(d[[a]], d[[b]], paired = TRUE, alternative = "two.sided", exact = FALSE)$p.value
}, a, b)]
comp_df_paired[, q := p.adjust(p_raw, method = "BH")]

assign_bracket_tiers <- function(comp_df, xpos) {
  comp_ord <- copy(comp_df)
  comp_ord[, x1 := xpos[a]]
  comp_ord[, x2 := xpos[b]]
  comp_ord[, left  := pmin(x1, x2)]
  comp_ord[, right := pmax(x1, x2)]
  comp_ord[, span  := right - left]
  setorder(comp_ord, span, q)

  levels_used <- list()
  comp_ord[, tier := 0L]

  for (i in seq_len(nrow(comp_ord))) {
    left  <- comp_ord$left[i]
    right <- comp_ord$right[i]
    tier  <- 1

    repeat {
      current <- if (tier <= length(levels_used)) levels_used[[tier]] else NULL
      overlaps <- FALSE

      if (!is.null(current)) {
        overlaps <- any(vapply(current, function(interval) {
          !(right < interval[1] || left > interval[2])
        }, logical(1)))
      }

      if (!overlaps) break
      tier <- tier + 1
    }

    comp_ord$tier[i] <- tier
    prior <- if (tier <= length(levels_used)) levels_used[[tier]] else list()
    levels_used[[tier]] <- c(prior, list(c(left, right)))
  }

  comp_ord
}

comp_ord <- assign_bracket_tiers(comp_df_paired, xpos)
n_tiers <- max(comp_ord$tier)
y_base  <- 1.08
y_step  <- 0.17
h       <- 0.02
y_max   <- y_base + (n_tiers - 1) * y_step + 0.14

plot_A <- ggplot(long_box, aes(method, cor, fill = method)) +
  geom_boxplot(width = 0.5, outlier.shape = NA, color = "black",
               linewidth = 0.25, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.35, shape = 21, fill = "white",
              color = "#333333", stroke = 0.15, alpha = 0.75) +
  geom_point(data = mean_df, aes(x = method, y = mean_cor), shape = 23,
             size = 1.4, fill = "white", color = "black", stroke = 0.4,
             inherit.aes = FALSE) +
  geom_text(data = mean_df,
            aes(x = method, y = max_cor + 0.05, label = sprintf("%.3f", mean_cor)),
            size = 3.2, fontface = "bold", inherit.aes = FALSE) +
  scale_fill_manual(values = METHOD_COLORS, na.value = "grey70") +
  scale_y_continuous(breaks = seq(0, 1, 0.25), expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max), clip = "on") +
  labs(x = NULL, y = "Max Pearson r per cell type") +
  theme_pub() +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 35, hjust = 1, size = AXIS_LABEL_SIZE),
        panel.grid.major = element_line(color = "grey88", linewidth = 0.25),
        panel.grid.minor = element_blank())

for (i in seq_len(nrow(comp_ord))) {
  row <- comp_ord[i, ]
  y   <- y_base + (row$tier - 1) * y_step

  if (
    (row$a == "CLAMPfull" & row$b == "CLAMPbase") |
    (row$a == "CLAMPbase" & row$b == "CLAMPfull")
  ) {
    y <- y + 0.02
  }

  lbl <- fmt_q(row$q)

  plot_A <- plot_A +
    annotate("segment", x = row$left,  xend = row$left,  y = y,       yend = y + h, linewidth = 0.25) +
    annotate("segment", x = row$left,  xend = row$right, y = y + h, yend = y + h, linewidth = 0.25) +
    annotate("segment", x = row$right, xend = row$right, y = y,       yend = y + h, linewidth = 0.25) +
    annotate("text",
             x = (row$left + row$right) / 2,
             y = y + h + 0.015,
             label = lbl,
             size = 3.0,
             vjust = 0,
             parse = TRUE,
             family = "sans")
}

options(repr.plot.width = 10, repr.plot.height = 3.2)
print(plot_A)


## Panel B: held-out test-prediction scatter, by dataset

In [ ]:
predictions          <- fread(snakemake@input[["holdout_predictions"]])
thresholded_metrics  <- fread(snakemake@input[["holdout_thresholded_metrics"]])

valid_keys <- thresholded_metrics[
  valid_all_folds == TRUE,
  .(dataset, method, cell_type)
]
scatter_data <- merge(predictions, valid_keys, by = c("dataset", "method", "cell_type"))
scatter_data[, panel_label := DATASET_LABELS[dataset]]

plot_statistics <- scatter_data[, {
  fit <- lm(predicted ~ observed)
  fit_summary <- summary(fit)
  t_val <- fit_summary$coefficients["observed", "t value"]
  df_resid <- fit_summary$df[2]
  # Two-sided p-value computed in log-space: for these huge n / huge t, the
  # linear p-value underflows to exactly 0, which would print as a nonsense
  # "0 x 10^0" if formatted directly.
  log10_p <- (log(2) + pt(-abs(t_val), df_resid, log.p = TRUE)) / log(10)
  list(
    pooled_r2 = fit_summary$r.squared,
    log10_p = log10_p
  )
}, by = panel_label]
setorder(plot_statistics, -pooled_r2)
panel_levels <- plot_statistics$panel_label
scatter_data[, panel_label := factor(panel_label, levels = panel_levels)]
plot_statistics[, panel_label := factor(panel_label, levels = panel_levels)]
# R^2 and p on two SEPARATE geom_text layers (not atop(), which centers
# the two lines relative to each other rather than left-aligning them) so
# both start at the same x. Standard R/statistics convention for an
# unreportably small p-value is "< 2.2e-16" (double-precision machine
# epsilon, e.g. what cor.test() reports) rather than an exact exponent.
log10_eps <- log10(2.2e-16)
plot_statistics[, r2_label := sprintf('R^2 == %.2f', pooled_r2)]
plot_statistics[, p_label := fifelse(
  log10_p < log10_eps,
  'italic(p) < 2.2 %*% 10^{-16}',
  sprintf('italic(p) == %.3f', 10^log10_p)
)]

plot_B <- ggplot(scatter_data, aes(x = observed, y = predicted)) +
  geom_abline(slope = 1, intercept = 0, color = "black", linetype = "dashed", linewidth = 0.35) +
  geom_point(aes(color = panel_label), alpha = 0.28, size = 0.22, show.legend = FALSE) +
  geom_text(
    data = plot_statistics,
    aes(x = 0.02, y = 0.98, label = r2_label),
    inherit.aes = FALSE, hjust = 0, vjust = 1, size = 2.2, parse = TRUE
  ) +
  geom_text(
    data = plot_statistics,
    aes(x = 0.02, y = 0.88, label = p_label),
    inherit.aes = FALSE, hjust = 0, vjust = 1, size = 2.2, parse = TRUE
  ) +
  facet_wrap(~panel_label, ncol = 3, scales = "free_x") +
  coord_cartesian(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE) +
  scale_x_continuous(
    breaks = seq(0, 1, 0.25),
    labels = c("0", "0.25", "0.50", "0.75", "1.00")
  ) +
  scale_y_continuous(breaks = seq(0, 1, 0.25),
                     labels = c("0", "0.25", "0.50", "0.75", "1.00")) +
  scale_color_manual(values = DATASET_COLORS) +
  labs(
    x = "True cell proportion in test sample",
    y = "Predicted cell proportion in test sample"
  ) +
  theme_pub() +
  theme(
    aspect.ratio = 1,
    strip.background = element_rect(fill = "grey94", color = "black", linewidth = 0.3),
    panel.spacing.x = grid::unit(1.4, "lines"),
    panel.spacing.y = grid::unit(0.6, "lines")
  )

options(repr.plot.width = 6, repr.plot.height = 8)
print(plot_B)


## Panel C: marker-recovery dot grid, by dataset

In [ ]:
module_ready <- fread(snakemake@input[["marker"]])

GRID_COLORS <- c("#b2182b", "#f4a582", "#f7f7f7", "#a1d99b", "#007a33")
GRID_VALUES <- c(0, 0.35, 0.5, 0.65, 1)

module_blocks <- lapply(split(module_ready, module_ready$dataset), function(d) {
  ds <- d$dataset[1]
  ds_label <- if (ds %in% names(DATASET_LABELS)) DATASET_LABELS[ds] else ds
  d[, marker_row_label := abbreviate_ct(marker_row_label)]
  d[, marker_row_label := wrap_label(marker_row_label, width = 18)]
  d[, lv_col_label     := abbreviate_ct(lv_col_label)]
  # Some LVs match 2 near-duplicate pathway terms (e.g. "OPC - OPC - LV11"
  # from "Oligodendrocyte progenitor/precursor cell" synonyms), joined by
  # 02_disentangle.ipynb (LABEL_MAX_TERMS=2). For the axis label, keep only
  # the single best (lowest-FDR, first) term -- the second is redundant and
  # was blowing labels up to 70+ chars, far past what any column width fits.
  d[, lv_col_label     := vapply(strsplit(lv_col_label, " - "), function(p) {
    if (length(p) >= 3) paste(p[1], p[length(p)], sep = " - ") else paste(p, collapse = " - ")
  }, character(1))]
  d[, lv_col_label     := wrap_label(lv_col_label, width = 14)]
  row_levels <- unique(d[order(marker_row_rank), marker_row_label])
  col_levels <- unique(d[order(lv_col_rank), lv_col_label])
  d[, marker_row_label := factor(marker_row_label, levels = row_levels)]
  d[, lv_col_label     := factor(lv_col_label, levels = col_levels)]
  diag_recovered <- d[diag_recovered == TRUE]
  Z_LIM     <- d$Z_LIM[1]
  Q_LIM     <- d$Q_LIM[1]
  n_samples <- d$n_samples[1]

  p <- ggplot(d, aes(x = lv_col_label, y = marker_row_label)) +
    geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
    { if (nrow(diag_recovered) > 0)
        geom_point(data = diag_recovered, aes(size = neg_log10_fdr),
                   shape = 1, color = "black", stroke = 0.8) } +
    scale_size_continuous(limits = c(0, Q_LIM), range = c(0.4, 3.2),
                          name = expression("-" * log[10] ~ "(FDR)")) +
    scale_color_gradientn(colors = GRID_COLORS, values = GRID_VALUES,
                          limits = c(-Z_LIM, Z_LIM), name = "LV effect") +
    coord_fixed() +
    scale_x_discrete(drop = FALSE) +
    scale_y_discrete(drop = FALSE) +
    theme_bw(base_size = 7.5) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 6),
          axis.text.y = element_text(size = 6.5),
          plot.title  = element_text(face = "bold", size = 8, hjust = 0.5)) +
    labs(x = NULL, y = NULL, title = paste0(ds_label, " (n = ", n_samples, ")"))

  list(plot = p, n_cat = length(row_levels))
})

# Organize by size (heaviest first, like panel D) rather than an arbitrary
# tissue order, so same-scale panels sit together within each row.
module_order  <- order(-vapply(module_blocks, `[[`, "n_cat", FUN.VALUE = numeric(1)))
module_blocks <- module_blocks[module_order]
module_panels <- lapply(module_blocks, `[[`, "plot")
# Panel width (hence rendered square size, under coord_fixed) scales with
# sqrt(n cell types), so panel AREA -- not just its linear dimension --
# doubles when a dataset's cell-type count doubles.
module_n_cat  <- vapply(module_blocks, `[[`, "n_cat", FUN.VALUE = numeric(1))
names(module_n_cat) <- names(module_blocks)
module_widths <- sqrt(module_n_cat)

# C's two rows: the 3 largest datasets together, the 3 smallest together --
# similar-sized panels share a row (rather than spreading the single
# biggest one out with the smallest ones), so each row is visually
# consistent in scale.
C_row1_idx <- seq_len(min(3, length(module_widths)))
C_row2_idx <- setdiff(seq_along(module_widths), C_row1_idx)

options(repr.plot.width = 20, repr.plot.height = 4)
print(wrap_plots(module_panels, ncol = 6, guides = "collect", widths = module_widths))


## Panel D: hard-to-distinguish cell-type dot grid

In [ ]:
hard_ready <- fread(snakemake@input[["hard"]])

group_dot_panels <- lapply(split(hard_ready, hard_ready$group_id), function(d) {
  d[, marker_row_label := abbreviate_ct(marker_row_label)]
  d[, marker_row_label := wrap_label(marker_row_label, width = 14)]
  d[, lv_col_label     := abbreviate_ct(lv_col_label)]
  d[, lv_col_label     := wrap_label(lv_col_label, width = 14)]
  row_levels <- unique(d[order(marker_row_rank), marker_row_label])
  col_levels <- unique(d[order(lv_col_rank), lv_col_label])
  d[, marker_row_label := factor(marker_row_label, levels = row_levels)]
  d[, lv_col_label     := factor(lv_col_label, levels = col_levels)]
  Z_LIM       <- d$Z_LIM[1]
  Q_LIM       <- d$Q_LIM[1]
  # Keep the original wrap_panel_title() line breaks (built upstream in
  # 04_hard_cell_types.ipynb) so long titles wrap instead of truncating.
  panel_title <- abbreviate_ct(d$panel_title[1])

  ggplot(d, aes(x = lv_col_label, y = marker_row_label)) +
    geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
    coord_fixed() +
    scale_size_continuous(limits = c(0, Q_LIM), range = c(0.4, 3.2),
                          name = expression("-" * log[10] ~ "(FDR)")) +
    scale_color_gradientn(colors = GRID_COLORS, values = GRID_VALUES,
                          limits = c(-Z_LIM, Z_LIM), name = "LV effect") +
    theme_bw(base_size = 7.5) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 6.5),
          axis.text.y = element_text(size = 6.5),
          axis.title  = element_blank(),
          plot.title  = element_text(face = "bold", size = 7, hjust = 0.5,
                                      lineheight = 0.95, margin = margin(b = 2)),
          plot.margin = margin(1, 1, 1, 1)) +
    labs(x = NULL, y = NULL, title = panel_title)
})

options(repr.plot.width = 20, repr.plot.height = 6)
print(wrap_plots(group_dot_panels, ncol = 2, guides = "collect"))


## Panel E: GTEx subtissue recovery from out-of-fold RF/SHAP

In [ ]:
subtissue_summary <- fread(snakemake@input[["subtissue_lr_summary"]])
subtissue_summary <- subtissue_summary[!is.na(SHAP_LR_Balanced_Accuracy)]
setorder(subtissue_summary, -SHAP_LR_Balanced_Accuracy)
tissue_order_E <- subtissue_summary$Tissue

subtissue_long <- melt(
  subtissue_summary,
  id.vars = c("Tissue", "Chance_Balanced_Accuracy", "RF_Passed_Accuracy_Gate"),
  measure.vars = c("SHAP_LR_Balanced_Accuracy", "FullLV_LR_Balanced_Accuracy"),
  variable.name = "model", value.name = "balanced_accuracy"
)
subtissue_long[, model := fifelse(model == "SHAP_LR_Balanced_Accuracy",
                                   "SHAP-derived LR", "Full-LV baseline LR")]
subtissue_long[, model := factor(model, levels = c("SHAP-derived LR", "Full-LV baseline LR"))]
subtissue_long[, Tissue := factor(Tissue, levels = tissue_order_E)]

# Flag tissues where the tissue-blind RF itself failed the accuracy gate
# with "*" on the x-axis -- their SHAP profile is less trustworthy to
# begin with, so the reader should discount those bars accordingly.
tissue_labels_E <- ifelse(subtissue_summary$RF_Passed_Accuracy_Gate,
                          subtissue_summary$Tissue, paste0(subtissue_summary$Tissue, "*"))
names(tissue_labels_E) <- subtissue_summary$Tissue

SUBTISSUE_MODEL_COLORS <- c("SHAP-derived LR" = CLAMP_COLOR, "Full-LV baseline LR" = "#DD8452")

plot_E <- ggplot(subtissue_long, aes(x = Tissue, y = balanced_accuracy, fill = model)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.65) +
  geom_errorbar(
    data = subtissue_summary,
    aes(x = Tissue, ymin = Chance_Balanced_Accuracy, ymax = Chance_Balanced_Accuracy),
    inherit.aes = FALSE, linetype = "22", color = "black", linewidth = 0.35, width = 0.75
  ) +
  scale_fill_manual(values = SUBTISSUE_MODEL_COLORS, name = NULL) +
  scale_x_discrete(labels = tissue_labels_E) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25),
                     expand = expansion(mult = c(0, 0.03))) +
  labs(x = NULL, y = "Balanced accuracy\n(subtissue recovery)") +
  theme_pub() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = AXIS_LABEL_SIZE),
    legend.position = "top",
    legend.direction = "horizontal"
  )

options(repr.plot.width = 6, repr.plot.height = 3.2)
print(plot_E)

## Panel F: xCell hepatocyte enrichment vs. liver SHAP importance

In [ ]:
liver_xcell <- fread(snakemake@input[["liver_xcell_scatter"]])

ct_test_liver <- cor.test(liver_xcell$shap_pct, liver_xcell$top1pct, method = "pearson")
r_val_liver <- unname(ct_test_liver$estimate)
p_val_liver <- ct_test_liver$p.value
fmt_p_liver <- if (p_val_liver < 0.001) {
  e_str <- formatC(p_val_liver, format = "e", digits = 2)
  parts <- strsplit(e_str, "e")[[1]]
  sprintf("italic(P) == %s %%*%% 10^{%d}", trimws(parts[1]), as.integer(parts[2]))
} else {
  sprintf("italic(P) == %.3f", p_val_liver)
}
stat_label_liver <- sprintf('atop(italic(r) == %.3f, %s)', r_val_liver, fmt_p_liver)

# LV21 and LV471 are the two highest-SHAP liver-classifying modules in this
# run (see accuracy_summary.tsv / liver_xcell_scatter.csv) -- highlighted
# directly rather than hardcoding an LV list disconnected from the data.
highlight_lvs <- liver_xcell[order(-shap_pct)][1:2, LV]
liver_xcell[, highlight := LV %in% highlight_lvs]

plot_F <- ggplot(liver_xcell, aes(x = shap_pct, y = top1pct)) +
  geom_point(aes(color = dominant_ct == "Hepatocytes"), size = 1.3, alpha = 0.8) +
  ggrepel::geom_text_repel(
    data = liver_xcell[highlight == TRUE],
    aes(label = LV), size = 2.4, fontface = "bold", color = "black",
    min.segment.length = 0, segment.size = 0.3, box.padding = 0.4
  ) +
  geom_text(
    data = data.table(x = Inf, y = Inf, label = stat_label_liver),
    aes(x = x, y = y, label = label), inherit.aes = FALSE,
    hjust = 1.05, vjust = 1.3, size = 2.6, parse = TRUE
  ) +
  scale_color_manual(values = c(`TRUE` = "#B2182B", `FALSE` = "grey60"),
                     labels = c(`TRUE` = "Hepatocyte-dominant", `FALSE` = "Other"),
                     name = NULL) +
  labs(x = "SHAP importance for liver classification (%)",
       y = "Mean hepatocyte xCell score\n(top 1% highest-activity samples)") +
  theme_pub() +
  theme(legend.position = "top", legend.direction = "horizontal")

options(repr.plot.width = 5, repr.plot.height = 3.2)
print(plot_F)

## Assembly and export

In [ ]:
# A single, fixed mm-per-sqrt(n_cat) unit applies to every one of the 6 C
# datasets, regardless of which row they land in -- panel size vs.
# cell-type count is the same ratio everywhere (homogeneous), rather than
# back-solving a unit to hit some other target width. Calibrated for
# legibility at the current axis-text size (6pt); if that font grows
# again, this needs to go up too, not down.
UNIT_PER_SQRT_NCAT <- 28.5

sum1 <- sum(module_widths[C_row1_idx]); max1 <- max(module_widths[C_row1_idx])
sum2 <- sum(module_widths[C_row2_idx]); max2 <- max(module_widths[C_row2_idx])
GLOBAL_UNIT <- UNIT_PER_SQRT_NCAT
C_width   <- GLOBAL_UNIT * sum1
C_row1_h  <- GLOBAL_UNIT * max1
C_row2_h  <- GLOBAL_UNIT * max2
H_C       <- C_row1_h + C_row2_h
row2_pad  <- sum1 - sum2

CD_GAP <- 14
# D: 2 cols x 3 rows of uniformly-sized square panels (coord_fixed), sized
# so its total height matches H_C -- a 2-wide grid needs width:height =
# 2:3 for square cells, considerably narrower than the old 3-wide grid at
# the same height, which is what makes D noticeably smaller now.
D_width <- H_C * (2 / 3)

FIG_W <- C_width + CD_GAP + D_width

# Row 1 (A + B): fixed 3:2 width ratio. B's facet grid is 3 cols x 2 rows
# with aspect.ratio = 1 per cell, so square cells need width:height = 3:2;
# ROW1_H is solved directly from B_width so B fills its box exactly.
A_width <- FIG_W * 3 / 5
B_width <- FIG_W * 2 / 5
ROW1_H  <- B_width / 1.5

# Legend strip: extracted once (C and D share identical LV-effect/FDR
# scales), made a bit bigger than the panel text, and centered on the
# FULL page width (not just the C/D seam).
LEGEND_H <- 14
legend_grob <- cowplot::get_legend(
  module_panels[[1]] +
    guides(color = guide_colorbar(order = 1), size = guide_legend(order = 2)) +
    theme(legend.position = "top",
          legend.direction = "horizontal",
          legend.box = "horizontal",
          legend.key.size = unit(3.2, "mm"),
          legend.text = element_text(size = LEGEND_TEXT_SIZE + 1.5),
          legend.title = element_text(size = LEGEND_TITLE_SIZE + 1.5))
)
legend_width <- 0.4
legend_row <- ggdraw() +
  draw_grob(legend_grob,
            x = 0.5 - legend_width / 2, y = 0,
            width = legend_width, height = 1)

# Top-align panels of different sizes within a row: stack each panel over
# a blank NULL spacer via cowplot::plot_grid(ncol = 1), sized so the panel
# occupies exactly its own weight's share of the row and the spacer takes
# the rest. This is a plain, literal vertical split (no shared grid/guide
# system to second-guess), so panels reliably start flush at the same top
# edge regardless of how their individual weights differ -- patchwork's
# area()-based t/b span trick looked equivalent on paper but did not
# actually keep titles aligned once row weights got far apart.
make_top_anchored_row <- function(panels, weights, pad_weight = 0) {
  row_max <- max(weights)
  cells <- lapply(seq_along(panels), function(i) {
    p <- panels[[i]] + theme(legend.position = "none")
    if (weights[i] < row_max) {
      plot_grid(p, NULL, ncol = 1,
                rel_heights = c(weights[i], row_max - weights[i]))
    } else {
      p
    }
  })
  if (pad_weight > 0) {
    cells   <- c(cells, list(NULL))
    weights <- c(weights, pad_weight)
  }
  plot_grid(plotlist = cells, nrow = 1, rel_widths = weights)
}

C_row1 <- make_top_anchored_row(module_panels[C_row1_idx], module_widths[C_row1_idx])
C_row2 <- make_top_anchored_row(module_panels[C_row2_idx], module_widths[C_row2_idx], pad_weight = row2_pad)

C_block <- plot_grid(
    C_row1, C_row2,
    ncol = 1, rel_heights = c(C_row1_h, C_row2_h)
)

# D: a smaller, uniformly-sized 2x3 zoom-in detail grid -- no separate
# caption or "D" tag (it reads as part of the same "C" panel, just a
# zoomed-in detail), just the plain grid at D_width.
D_block <- wrap_plots(group_dot_panels, ncol = 2) &
    theme(legend.position = "none")

# A simple zoom-in connector in the C/D gap: two lines converging from the
# vertical middle of C's edge out to D's top-left/bottom-left corners,
# reading as "this expands into the detail views on the right" without
# tracking any specific cell's coordinates (which would be fragile).
zoom_connector <- ggplot() +
  geom_segment(aes(x = 0, xend = 1, y = 0.5, yend = 1),
               linewidth = 0.4, color = "grey50", linetype = "22") +
  geom_segment(aes(x = 0, xend = 1, y = 0.5, yend = 0),
               linewidth = 0.4, color = "grey50", linetype = "22") +
  scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
  scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
  theme_void()

row_CD <- plot_grid(
    C_block, zoom_connector, D_block,
    ncol = 3, rel_widths = c(C_width, CD_GAP, D_width),
    labels = c("C", "", ""),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

# Row 3 (E + F): same 3:2 width split and same height as row 1, so the two
# "wide plain-panel" rows (A+B, E+F) read as a matched pair bracketing the
# dot-grid block (C/D) in between.
E_width  <- FIG_W * 3 / 5
F_width  <- FIG_W * 2 / 5
ROW_EF_H <- ROW1_H

FIG_H <- ROW1_H + H_C + ROW_EF_H + LEGEND_H

row1 <- plot_grid(
    plot_A, plot_B,
    ncol = 2, rel_widths = c(A_width, B_width), align = "h", axis = "tb",
    labels = c("A", "B"),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

row_EF <- plot_grid(
    plot_E, plot_F,
    ncol = 2, rel_widths = c(E_width, F_width), align = "h", axis = "tb",
    labels = c("E", "F"),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

fig2 <- plot_grid(
    row1, row_CD, row_EF, legend_row,
    ncol = 1, rel_heights = c(ROW1_H, H_C, ROW_EF_H, LEGEND_H)
)

ggsave(snakemake@output[["pdf"]], fig2,
       width = FIG_W, height = FIG_H, units = "mm",
       device = cairo_pdf, bg = "white")
ggsave(snakemake@output[["png"]], fig2,
       width = FIG_W, height = FIG_H, units = "mm",
       dpi = 300, bg = "white")
ggsave(snakemake@output[["svg"]], fig2,
       width = FIG_W, height = FIG_H, units = "mm",
       device = svglite::svglite, bg = "white")
cat("fig2 exported to", dirname(snakemake@output[["png"]]), "\n")
cat("FIG_W =", FIG_W, "mm, FIG_H =", FIG_H, "mm, C_width =", C_width, "D_width =", D_width, "\n")

# The inline display device inherits whatever the LAST options(repr.plot.*)
# call set (Panel F's small 5x3.2in), far smaller than this full 5-row
# composite -- rendering it into that tiny device produced "Viewport has
# zero dimension(s)" from some nested sub-viewport. Size the device to the
# actual composite (mm -> in) before the auto-print below.
options(repr.plot.width = FIG_W / 25.4, repr.plot.height = FIG_H / 25.4)
fig2
